# Module B · N2 — The frozen Candidate-V1 benchmark on FINAL-01Decision D5 (15 Sep 2026): when the new dataset lands, rerun the same benchmarkwith **no configuration changes**, so any difference is attributable to the datarather than to model shopping.Nothing in `moduleb.config`'s `FROZEN_V1` block has been touched to producethese numbers. The readable version of `scripts/03_benchmark_frozen_v1.py`.

In [ ]:
import sys, pathlibROOT = pathlib.Path.cwd().parent if pathlib.Path.cwd().name == "notebooks" else pathlib.Path.cwd()sys.path.insert(0, str(ROOT))import numpy as np, pandas as pdpd.set_option("display.width", 200); pd.set_option("display.max_columns", 80)from moduleb import (baselines, config, contract, cv, dataio, envelope,                     freeze, guards, metrics, models, reason_codes)from moduleb.constants import PARAMS, TARGET_COLSfrom moduleb.features import add_features, make_foldsprint("moduleb ready — frozen config digest", config.frozen_config_digest()[:16])

## 1. The configuration being run

In [ ]:
display(pd.DataFrame(config.RECOMMENDED).T.rename_axis("param"))print("Huber:", config.HUBER)print("folds:", config.N_FOLDS, " seed:", config.GLOBAL_SEED)print("digest:", config.frozen_config_digest())

## 2. Whole-lot cross-validationThe fold *is* the lot. `cross_validate` asserts that on every pass, so an editthat reintroduced row-level splitting would fail the run rather than produce anoptimistic table.

In [ ]:
tr = dataio.load_split("train"); base, limits = dataio.load_specs()feat = add_features(tr.frame, base); feat["fold"] = make_folds(feat, config.N_FOLDS)res, oof = cv.cross_validate(feat, config.RECOMMENDED, with_baselines=True)piv = (res[res.variant == "ALL"].pivot(index="param", columns="model", values="MAE")       .loc[PARAMS, ["Persistence", "LinExtrap", "MedianRatio_24h", "MODULE_B"]])piv["gain_vs_median_ratio_pct"] = 100*(piv.MedianRatio_24h - piv.MODULE_B)/piv.MedianRatio_24hdisplay(piv.round(5))

## 3. The paired per-lot test — the number to quoteHeld-out **lots** are the unit of independence, not rows. Components in a lotshare whatever that lot did, so treating 3,151 rows as 3,151 independentobservations would overstate significance by roughly the lot size.The team rule: differences under 5% MAE are **ties**, whatever the rank or thep-value.

In [ ]:
rows = []for p in PARAMS:    mr = baselines.fold_baselines(feat, p, feat.fold.to_numpy())["MedianRatio_24h"]    t = metrics.paired_lot_test(feat[f"{p}_168h"].to_numpy(float), oof[p], mr,                                feat.lot_id.to_numpy())    t.update(param=p, verdict=metrics.verdict(t, config.TIE_THRESHOLD_PCT,                                              config.PAIRED_TEST_ALPHA))    rows.append(t)display(pd.DataFrame(rows)[["param","mae_a","mae_b","gain_pct","lots_won","n_lots",                            "wilcoxon_p","verdict"]].round(5))

## 4. Bias — the direction that matters`MeanSignedError < 0` means the forecast is systematically **low**. In ascreening context that is the dangerous direction, so it is reported next to theMAE rather than buried in a results file.

In [ ]:
display(res[(res.variant=="ALL") & (res.model.isin(["MODULE_B","MedianRatio_24h"]))]        [["param","model","MAE","MedAE","P90AE","RMSE","MeanSignedError",          "UnderPredRate","MacroLotMAE","WorstLotMAE"]].round(5))

## 5. The configuration grid — evidence only`scripts/04_benchmark_grid.py` scores 24 configurations on the same folds. It isa small enumerable grid, not a hyper-parameter search: blind optimisation over 42lots finds fold noise, and the 5% tie rule already says so.It takes about six minutes. Run it from the terminal and read`results/04_best_vs_frozen.csv` — nothing there is ever applied automatically.

In [ ]:
p = dataio.RESULTS / "04_best_vs_frozen.csv"display(pd.read_csv(p).round(5) if p.exists()        else "run: python scripts/04_benchmark_grid.py")

Next: **N3_calibration.ipynb**.